Notebook to use Yahoo data for creating universe and returns

In [ ]:
# Importing libraries
import pandas as pd
import requests
from io import BytesIO
import os

In [ ]:
# Create stores folder if doesn't exist

BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "stores")

os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
url = "https://github.com/yszanwar/phase2_qrt_challenge/releases/download/price_data/all_prices_5000_tickers.parquet"

response = requests.get(url)
pv = pd.read_parquet(BytesIO(response.content), engine="pyarrow")
print(f"Data loaded. Shape: {pv.shape}")

In [ ]:
# Calculate Average Daily Volume for trailing 20 days
df_daily_volume = pv['Close'].mul(pv['Volume']).fillna(0)
df_average_daily_volume = df_daily_volume.rolling(window=20, min_periods=1).mean()

In [ ]:
# Creating 5M+ Average Daily Volume Universe
import matplotlib.pyplot as plt

df_universe_5m = (df_average_daily_volume > 5_000_000).astype(int)
df_universe_5m.sum(axis=1).plot(figsize=(12, 6))
plt.title("Number of Stocks with Average Daily Volume > $5M")
plt.xlabel("Date")
plt.ylabel("Number of Stocks")
plt.show()

# Saving 5M universe to stores folder
df_universe_5m.to_parquet(os.path.join(DATA_DIR, "universe_5m.parquet"), engine="pyarrow")
print(f"5M universe saved. Latest count: {df_universe_5m.iloc[-1].sum()}")

In [ ]:
# Creating 1M+ Average Daily Volume Universe
df_universe_1m = (df_average_daily_volume > 1_000_000).astype(int)
df_universe_1m.sum(axis=1).plot(figsize=(12, 6))
plt.title("Number of Stocks with Average Daily Volume > $1M")
plt.xlabel("Date")
plt.ylabel("Number of Stocks")
plt.show()

# Saving 1M universe to parquet file
df_universe_1m.to_parquet(os.path.join(DATA_DIR, "universe_1m.parquet"), engine="pyarrow")
print(f"1M universe saved. Latest count: {df_universe_1m.iloc[-1].sum()}")

In [ ]:
# Calculating returns for each ticker every day
returns = pv['Adj Close'].pct_change(fill_method=None).fillna(0)

# Saving returns to parquet file
returns.to_parquet(os.path.join(DATA_DIR, "returns.parquet"), engine="pyarrow")
print(f"Returns saved. Shape: {returns.shape}")

In [ ]:
# Verify saved files
print("\n" + "="*50)
print("VERIFICATION - FILES SAVED")
print("="*50)

files = {
    "5M Universe": "universe_5m.parquet",
    "1M Universe": "universe_1m.parquet",
    "Returns": "returns.parquet"
}

for name, filename in files.items():
    filepath = os.path.join(DATA_DIR, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath) / (1024 * 1024)
        print(f"✓ {name}: {filename} ({size:.2f} MB)")
    else:
        print(f"✗ {name}: {filename} (NOT FOUND)")

print("\n" + "="*50)
print("NOTEBOOK COMPLETE")
print("="*50)